# Sesión 7 — MBD

## Clasificación supervisada (II)

Clasificamos productos con **árboles de inferencia condicional**, **random forest** y **SVM**.
Seguimos el recorrido de la sesión: representación, poda, evaluación OOB, importancia de variables
y ajuste de hiperparámetros. Las prácticas 7-1, 7-2 y 7-3 quedan para resolver.

En VS Code, selecciona un kernel de Python y ejecuta en orden con **Shift + Enter**.
Abre **Variables** para inspeccionar tablas y modelos, y **Esquema / Outline** para navegar.
Los controles aparecen al ejecutar sus celdas y siguen disponibles al avanzar por la sesión.
Usa **Reiniciar** y **Ejecutar todo** para repetirla desde cero.
[Guía de notebooks en VS Code](https://code.visualstudio.com/docs/datascience/jupyter-notebooks).

### Preparación

Limpiamos las variables y cargamos las librerías.
Si falta alguna en el kernel seleccionado, descomenta y ejecuta la instalación una vez.
Las funciones de apoyo se incluyen en el notebook y sus celdas se pueden desplegar.

In [ ]:
%reset -f

In [ ]:
# %pip install numpy pandas matplotlib scipy scikit-learn ipywidgets threadpoolctl ipykernel

In [ ]:
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from matplotlib.patches import FancyBboxPatch
from scipy import stats
from scipy.special import gammaln
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, GridSearchCV
from threadpoolctl import threadpool_limits

%matplotlib inline
%config InteractiveShell.ast_node_interactivity = "all"
plt.rcParams.update({"figure.figsize": (8, 4), "figure.dpi": 100})
pd.set_option("display.max_columns", 30)
SEED = 12345

In [ ]:
for carpeta in (Path.cwd(), *Path.cwd().parents):
    if (carpeta / "datos" / "products.csv").is_file():
        DATA_PATH = carpeta / "datos"
        break
else:
    raise FileNotFoundError("Abre el notebook dentro de la carpeta LaSalle_Estadistica.")

DATA_PATH

### Datos y variables

| Variable | Contenido |
| --- | --- |
| `id` | Identificador del producto |
| `feat_1` … `feat_93` | Características del producto |
| `target` | Clase del producto |

Leemos el archivo separado por punto y coma. Retiramos `id` antes de ajustar los modelos.

In [ ]:
datos = pd.read_csv(DATA_PATH / "products.csv", sep=";", decimal=",")
datos.shape
datos.head()

In [ ]:
datos.describe(include="all")
datos["target"].value_counts().sort_index()

### Partición de entrenamiento y test

Cada fila entra en entrenamiento con probabilidad 0,7. La semilla permite repetir la partición.
Árboles y bosques comparten estas filas. El bloque de SVM tendrá su propia submuestra y partición.

In [ ]:
d = datos.drop(columns="id")
predictores = list(d.columns.drop("target"))
clases = np.sort(d["target"].unique())
rng_arbol = np.random.default_rng(SEED)
train_sel = rng_arbol.choice([False, True], size=len(d), p=[0.3, 0.7])
train = d.loc[train_sel].copy()
test = d.loc[~train_sel].copy()
X_train, y_train = train[predictores], train["target"]
X_test, y_test = test[predictores], test["target"]
pd.Series({"Entrenamiento": len(train), "Test": len(test), "Predictores": len(predictores)})

Las matrices de confusión muestran **predicciones en las filas** y **clases reales en las columnas**.
El acierto por clase divide la diagonal por el número real de observaciones de cada clase.

In [ ]:
def confusion(reales, predichas, clases):
    return pd.crosstab(pd.Series(np.asarray(predichas), name="Predicción"),
                       pd.Series(np.asarray(reales), name="Real")).reindex(
                       index=clases, columns=clases, fill_value=0).rename_axis(index="Predicción", columns="Real")


def evaluar_clasificador(reales, predichas, clases):
    tabla = confusion(reales, predichas, clases)
    acierto = np.mean(np.asarray(reales) == np.asarray(predichas))
    por_clase = pd.Series(np.divide(np.diag(tabla), tabla.sum(axis=0),
                                  out=np.full(len(clases), np.nan), where=tabla.sum(axis=0) > 0),
                          index=clases, name="Acierto por clase")
    return SimpleNamespace(confusion=tabla, acierto=acierto, por_clase=por_clase)


def barras_clases(valores, titulo, ylabel="Proporción de aciertos"):
    fig, ax = plt.subplots()
    ax.bar(valores.index.astype(str), valores)
    ax.set(ylim=(0, 1), ylabel=ylabel, title=titulo)
    plt.setp(ax.get_xticklabels(), rotation=40, ha="right")
    fig.tight_layout()
    plt.show()

## 1. Árboles de inferencia condicional

En cada nodo contrastamos la asociación de cada predictor con la respuesta mediante un estadístico
cuadrático y su distribución asintótica. Ajustamos los contrastes por el conjunto de variables.
Si existe asociación significativa, escogemos una variable y buscamos un corte binario.

La implementación incluida está preparada para **predictores numéricos y una respuesta nominal**,
como los de este archivo. Permite limitar la profundidad, los tamaños de nodo y el nivel de significación.
Guardamos los p-valores en escala logarítmica para trabajar con valores muy pequeños.
[Fundamento de la inferencia condicional](https://www.zeileis.org/papers/Hothorn%2BHornik%2BZeileis-2006.pdf).

In [ ]:
def log_cola_chi2(estadistico, grados):
    valores = np.asarray(estadistico, dtype=float)
    logp = stats.chi2.logsf(valores, grados)
    extremos = ~np.isfinite(logp) & (valores > 0)
    if extremos.any():
        x, a = valores[extremos]/2, grados/2
        termino, suma = np.ones(len(x)), np.ones(len(x))
        for j in range(1, 12):
            termino *= (a-j)/x
            suma += termino
        logp[extremos] = -x + (a-1)*np.log(x) - gammaln(a) + np.log(suma)
    return logp


class ArbolNumerico:
    """Predicción y vistas comunes a árboles binarios con predictores numéricos."""

    def hojas(self, podados=None, profundidad=None):
        podados = set() if podados is None else set(podados)
        resultado = []
        def visitar(i):
            nodo = self.nodos[i]
            if nodo.izquierda is None or i in podados or (profundidad is not None and nodo.profundidad >= profundidad):
                resultado.append(i)
            else:
                visitar(nodo.izquierda)
                visitar(nodo.derecha)
        visitar(0)
        return resultado

    def predict_proba(self, X, podados=None, profundidad=None):
        valores = X.loc[:, self.variables].to_numpy(dtype=float)
        resultado = np.zeros((len(X), len(self.clases)))
        podados = set() if podados is None else set(podados)
        def visitar(i, filas):
            if not len(filas):
                return
            nodo = self.nodos[i]
            if nodo.izquierda is None or i in podados or (profundidad is not None and nodo.profundidad >= profundidad):
                resultado[filas] = nodo.recuentos/nodo.recuentos.sum()
            else:
                lado = valores[filas, nodo.variable] <= nodo.corte
                visitar(nodo.izquierda, filas[lado])
                visitar(nodo.derecha, filas[~lado])
        visitar(0, np.arange(len(X)))
        return pd.DataFrame(resultado, columns=self.clases, index=X.index)

    def predict(self, X, podados=None, profundidad=None):
        return self.predict_proba(X, podados, profundidad).idxmax(axis=1).rename("Predicción")

    def tabla_nodos(self):
        return pd.DataFrame([{
            "Nodo": i, "Profundidad": n.profundidad, "n": int(n.recuentos.sum()),
            "Variable": self.variables[n.variable] if n.variable is not None else None,
            "Corte": n.corte, "Clase": self.clases[n.recuentos.argmax()],
            "log10(p)": n.logp/np.log(10) if n.logp is not None else np.nan,
            "Terminal": n.izquierda is None}
            for i, n in enumerate(self.nodos)]).set_index("Nodo")


class ArbolCondicional(ArbolNumerico):
    """Inferencia condicional para predictores numéricos y respuesta nominal."""

    def __init__(self, max_depth=None, min_samples_split=20, min_samples_leaf=7,
                 alpha=0.05, min_probability=0.01, splitstat="maximum"):
        self.max_depth, self.min_samples_split = max_depth, min_samples_split
        self.min_samples_leaf, self.alpha = min_samples_leaf, alpha
        self.min_probability, self.splitstat = min_probability, splitstat

    def fit(self, X, y):
        self.variables = list(X.columns)
        self.clases, codigos = np.unique(y, return_inverse=True)
        valores = X.to_numpy(dtype=float)
        if not np.isfinite(valores).all():
            raise ValueError("Esta implementación requiere predictores numéricos sin ausentes.")
        self.nodos = []
        def construir(filas, profundidad):
            A, respuesta = valores[filas], codigos[filas]
            n = len(filas)
            recuentos = np.bincount(respuesta, minlength=len(self.clases)).astype(float)
            nodo = SimpleNamespace(profundidad=profundidad, recuentos=recuentos,
                                   variable=None, corte=np.nan, logp=None, izquierda=None, derecha=None)
            indice = len(self.nodos)
            self.nodos.append(nodo)
            presentes = recuentos > 0
            if presentes.sum() < 2 or n < self.min_samples_split or (self.max_depth is not None and profundidad >= self.max_depth):
                return indice
            centradas = A-A.mean(axis=0)
            ss_total = np.einsum("ij,ij->j", centradas, centradas)
            candidatas = ss_total > 1e-12
            if not candidatas.any():
                return indice
            sumas_clase = np.vstack([centradas[respuesta == c].sum(axis=0)
                                     for c in np.flatnonzero(presentes)])
            ss_entre = (sumas_clase**2/recuentos[presentes, None]).sum(axis=0)
            estadisticos = np.divide((n-1)*ss_entre, ss_total,
                                     out=np.zeros(len(self.variables)), where=candidatas)
            logp = log_cola_chi2(estadisticos, presentes.sum()-1)
            # Ajuste de la familia de contrastes: 1-(1-p)^m, evaluado con estabilidad.
            ajustados = logp.copy()
            pequenos = logp < -20
            ajustados[pequenos] += np.log(candidatas.sum())
            with np.errstate(divide="ignore", invalid="ignore"):
                ajustados[~pequenos] = np.log(-np.expm1(candidatas.sum()*np.log1p(-np.exp(logp[~pequenos]))))
            ajustados[~candidatas] = 0
            nodo.logp = float(ajustados.min())
            if nodo.logp > np.log(self.alpha):
                return indice
            minimo = max(self.min_samples_leaf, int(np.floor(n*self.min_probability))+1)
            for variable in np.argsort(ajustados, kind="stable")[:2]:
                if ajustados[variable] > np.log(self.alpha):
                    continue
                orden = np.argsort(A[:, variable], kind="stable")
                x = A[orden, variable]
                posiciones = np.arange(1, n)
                validas = (x[:-1] < x[1:]) & (posiciones >= minimo) & (n-posiciones >= minimo)
                if not validas.any():
                    continue
                izquierda = np.eye(len(self.clases))[respuesta[orden]].cumsum(axis=0)[:-1, presentes]
                proporcion = recuentos[presentes]/n
                diferencias = izquierda-posiciones[:, None]*proporcion
                if self.splitstat == "maximum":
                    varianza = posiciones[:, None]*(n-posiciones[:, None])/(n-1)*proporcion*(1-proporcion)
                    puntuacion = (diferencias**2/varianza).max(axis=1)
                elif self.splitstat == "quadratic":
                    puntuacion = (n-1)*(diferencias**2/proporcion).sum(axis=1)/(posiciones*(n-posiciones))
                else:
                    raise ValueError("splitstat debe ser maximum o quadratic.")
                puntuacion[~validas] = -np.inf
                posicion = int(np.argmax(puntuacion))
                nodo.variable, nodo.corte = int(variable), float(x[posicion])
                nodo.logp = float(ajustados[variable])
                division = A[:, variable] <= nodo.corte
                nodo.izquierda = construir(filas[division], profundidad+1)
                nodo.derecha = construir(filas[~division], profundidad+1)
                break
            return indice
        with threadpool_limits(limits=1):
            construir(np.arange(len(X)), 0)
        return self


def dibujar_arbol(modelo, profundidad=3, extendido=False, podados=None, titulo="Árbol de clasificación"):
    podados = set() if podados is None else set(podados)
    terminales = modelo.hojas(podados, profundidad)
    x_hoja = {n: i*1.7 for i, n in enumerate(terminales)}
    posiciones = {}
    conexiones = []
    def situar(i):
        nodo = modelo.nodos[i]
        if i in x_hoja:
            x = x_hoja[i]
        else:
            izq, der = situar(nodo.izquierda), situar(nodo.derecha)
            x = (izq+der)/2
            conexiones.extend([(i, nodo.izquierda, "≤"), (i, nodo.derecha, ">")])
        posiciones[i] = (x, -nodo.profundidad*1.6)
        return x
    situar(0)
    fig, ax = plt.subplots(figsize=(max(9, len(terminales)*1.55), max(4, (profundidad+1)*1.5)))
    colores = plt.get_cmap("tab10")
    for padre, hijo, signo in conexiones:
        xp, yp = posiciones[padre]
        xh, yh = posiciones[hijo]
        ax.plot([xp,xh],[yp-.3,yh+.3], color="0.6", zorder=0)
        ax.text((xp+xh)/2, (yp+yh)/2, signo, fontsize=10, ha="center", backgroundcolor="white")
    for i, (x,y) in posiciones.items():
        nodo = modelo.nodos[i]
        clase = int(nodo.recuentos.argmax())
        terminal = i in x_hoja
        color = colores(clase) if terminal else "#edf2f7"
        caja = FancyBboxPatch((x-.72,y-.36),1.44,.72,boxstyle="round,pad=0.04",facecolor=color,
                              alpha=.22 if terminal else 1,edgecolor="0.35")
        ax.add_patch(caja)
        total = int(nodo.recuentos.sum())
        if terminal:
            texto = f"Nodo {i} · n={total}\n{modelo.clases[clase]} ({nodo.recuentos[clase]/total:.0%})"
            ax.text(x,y+.1 if extendido else y,texto,ha="center",va="center",fontsize=9)
            if extendido:
                proporciones = nodo.recuentos/total
                ancho = 1.18/len(proporciones)
                ax.bar(x-.59+np.arange(len(proporciones))*ancho, proporciones*.23,
                       bottom=y-.29, width=ancho*.9, color=[colores(j) for j in range(len(proporciones))], align="edge")
        else:
            texto = f"Nodo {i} · n={total}\n{modelo.variables[nodo.variable]} ≤ {nodo.corte:g}"
            if nodo.logp is not None:
                texto += f"\nlog₁₀(p)={nodo.logp/np.log(10):.1f}"
            ax.text(x,y,texto,ha="center",va="center",fontsize=9)
    ys = [p[1] for p in posiciones.values()]
    ax.set(xlim=(-1, max(x_hoja.values())+1), ylim=(min(ys)-.65,.8), title=titulo)
    ax.axis("off")
    if extendido:
        leyenda = [plt.Line2D([0],[0],marker="s",linestyle="",color=colores(j),label=str(clase))
                   for j, clase in enumerate(modelo.clases)]
        fig.legend(handles=leyenda,loc="lower center",ncol=min(9,len(leyenda)),fontsize=8)
    fig.tight_layout(rect=(0,.04 if extendido else 0,1,1))
    plt.show()

### Construir un árbol poco profundo

Limitamos la profundidad a tres niveles. La variable se selecciona mediante el contraste;
el corte maximiza la separación estandarizada entre los dos grupos.

In [ ]:
ct_visual = ArbolCondicional(max_depth=3, splitstat="maximum").fit(X_train, y_train)
ct_visual.tabla_nodos()

#### Representación extendida: frecuencias de las clases en las hojas

In [ ]:
dibujar_arbol(ct_visual, profundidad=3, extendido=True, titulo="Árbol condicional · representación extendida")

#### Representación simple: reglas y clase asignada

In [ ]:
dibujar_arbol(ct_visual, profundidad=3, extendido=False, titulo="Árbol condicional · reglas de clasificación")

### Capacidad predictiva con profundidad sin límite explícito

In [ ]:
ct_mod = ArbolCondicional(splitstat="maximum").fit(X_train, y_train)
pred_ct = ct_mod.predict(X_test)
evaluacion_ct = evaluar_clasificador(y_test, pred_ct, clases)
t_ct = evaluacion_ct.confusion
t_ct
evaluacion_ct.acierto
pd.Series({"Nodos": len(ct_mod.nodos), "Hojas": len(ct_mod.hojas()),
           "Profundidad": max(n.profundidad for n in ct_mod.nodos)})

### Capacidad predictiva por clase

In [ ]:
evaluacion_ct.por_clase
barras_clases(evaluacion_ct.por_clase, "Árbol condicional · acierto por clase")

#### Explorar la representación

El control limita la profundidad que se dibuja. Los nodos colapsados muestran la clase mayoritaria
y las frecuencias que contienen; la predicción del modelo completo se conserva en `pred_ct`.

In [ ]:
def explorar_arbol(profundidad, extendido):
    dibujar_arbol(ct_mod, profundidad=profundidad, extendido=extendido,
                  titulo=f"Árbol condicional · vista hasta profundidad {profundidad}")

control_arbol = widgets.interactive(explorar_arbol,
    profundidad=widgets.IntSlider(value=3, min=1, max=4, description="Profundidad", continuous_update=False),
    extendido=widgets.Checkbox(value=True, description="Frecuencias en hojas"))
display(control_arbol)

#### Inspeccionar un nodo

In [ ]:
def explorar_nodo(nodo):
    detalle = ct_mod.nodos[nodo]
    display(ct_mod.tabla_nodos().loc[[nodo]])
    display(pd.DataFrame({"Frecuencia": detalle.recuentos.astype(int),
                          "Proporción": detalle.recuentos/detalle.recuentos.sum()}, index=clases))

control_nodo = widgets.interactive(explorar_nodo,
    nodo=widgets.IntSlider(value=0, min=0, max=len(ct_mod.nodos)-1, description="Nodo", continuous_update=False))
display(control_nodo)

### Poda manual por significación

Para este ajuste utilizamos un estadístico cuadrático también al escoger el corte.
Extraemos los nodos internos y los p-valores de sus contrastes.
Podar un nodo convierte ese nodo en hoja y descarta todo lo que cuelga de él.

In [ ]:
ct_poda = ArbolCondicional(splitstat="quadratic").fit(X_train, y_train)
nodos_ct = ct_poda.tabla_nodos()
internos_ct = nodos_ct.loc[~nodos_ct["Terminal"]].copy()
internos_ct[["Variable", "log10(p)"]]
internos_ct["log10(p)"].quantile(np.arange(0, 1.01, 0.1))

Retiramos los subárboles cuyos nodos cumplen `log10(p) > −95`.

In [ ]:
ids_poda = internos_ct.index[internos_ct["log10(p)"] > -95].tolist()
pred_ct_poda = ct_poda.predict(X_test, podados=ids_poda)
evaluacion_ct_poda = evaluar_clasificador(y_test, pred_ct_poda, clases)
t_ct_poda = evaluacion_ct_poda.confusion
t_ct_poda
evaluacion_ct_poda.acierto
pd.Series({"Hojas tras la poda": len(ct_poda.hojas(ids_poda)), "Predicciones ausentes": pred_ct_poda.isna().sum()})

In [ ]:
def explorar_poda_condicional(log10_umbral):
    podados = internos_ct.index[internos_ct["log10(p)"] > log10_umbral].tolist()
    prediccion = ct_poda.predict(X_test, podados=podados)
    resultado = evaluar_clasificador(y_test, prediccion, clases)
    display(pd.Series({"Hojas": len(ct_poda.hojas(podados)), "Proporción de aciertos": resultado.acierto}))
    display(resultado.confusion)
    dibujar_arbol(ct_poda, profundidad=3, extendido=False, podados=podados,
                  titulo=f"Poda por log₁₀(p) > {log10_umbral}")

control_poda = widgets.interactive(explorar_poda_condicional,
    log10_umbral=widgets.IntSlider(value=-95, min=-200, max=0, step=5,
                                  description="log₁₀(p)", continuous_update=False))
display(control_poda)

### Poda por complejidad de coste y error de clasificación

Construimos un árbol mediante entropía con un mínimo de 10 observaciones para dividir y 5 por hoja.
Dejamos de expandir los nodos cuya desviación es inferior al 1 % de la del nodo raíz.
Después calculamos una secuencia de subárboles, retirando las ramas con menor incremento de errores por hoja eliminada.
La pérdida utilizada para podar es el **número de errores de clasificación**.

In [ ]:
def arbol_entropia(X, y):
    ajuste = DecisionTreeClassifier(criterion="entropy", min_samples_split=10,
                                    min_samples_leaf=5, random_state=SEED).fit(X, y)
    modelo = ArbolNumerico()
    modelo.variables, modelo.clases, modelo.nodos = list(X.columns), ajuste.classes_, []
    t = ajuste.tree_
    umbral = .01*t.weighted_n_node_samples[0]*t.impurity[0]
    def copiar(i, profundidad):
        recuentos = t.value[i,0].astype(float)
        recuentos = np.rint(recuentos/recuentos.sum()*t.weighted_n_node_samples[i])
        nodo = SimpleNamespace(profundidad=profundidad, recuentos=recuentos, variable=None,
                               corte=np.nan, logp=None, izquierda=None, derecha=None)
        indice = len(modelo.nodos)
        modelo.nodos.append(nodo)
        if t.children_left[i] != t.children_right[i] and t.weighted_n_node_samples[i]*t.impurity[i] >= umbral:
            nodo.variable, nodo.corte = int(t.feature[i]), float(t.threshold[i])
            nodo.izquierda = copiar(t.children_left[i], profundidad+1)
            nodo.derecha = copiar(t.children_right[i], profundidad+1)
        return indice
    copiar(0, 0)
    return modelo


def trayectoria_poda(modelo):
    """Secuencia de complejidad de coste con pérdida de clasificación 0–1."""
    podados = set()
    filas = []
    alpha = -np.inf
    while True:
        candidatos = {}
        def recorrer(i):
            nodo = modelo.nodos[i]
            error_nodo = nodo.recuentos.sum()-nodo.recuentos.max()
            if nodo.izquierda is None or i in podados:
                return error_nodo, 1
            e1, h1 = recorrer(nodo.izquierda)
            e2, h2 = recorrer(nodo.derecha)
            candidatos[i] = (error_nodo-e1-e2)/(h1+h2-1)
            return e1+e2, h1+h2
        errores, hojas = recorrer(0)
        filas.append({"Hojas": hojas, "alpha": alpha, "Errores train": errores,
                      "podados": frozenset(podados)})
        if not candidatos:
            break
        alpha = max(0.0, alpha, min(candidatos.values()))
        podados.update(i for i, a in candidatos.items() if a <= alpha+1e-10)
    return pd.DataFrame(filas)


def podar_a_hojas(trayectoria, numero=10):
    posibles = trayectoria.loc[trayectoria["Hojas"] >= numero]
    return (posibles.iloc[-1] if len(posibles) else trayectoria.iloc[0])["podados"]


def validar_poda(X, y, trayectoria, cv=10):
    errores = np.zeros((cv, len(trayectoria)))
    particiones = KFold(n_splits=cv, shuffle=True, random_state=SEED)
    for vuelta, (entrenar, validar) in enumerate(particiones.split(X)):
        modelo = arbol_entropia(X.iloc[entrenar], y.iloc[entrenar])
        camino = trayectoria_poda(modelo)
        for j, alpha in enumerate(trayectoria["alpha"]):
            disponibles = camino.loc[camino["alpha"] <= alpha+1e-10] if np.isfinite(alpha) else camino.iloc[:1]
            prediccion = modelo.predict(X.iloc[validar], podados=disponibles.iloc[-1]["podados"])
            errores[vuelta,j] = np.mean(prediccion.to_numpy() != y.iloc[validar].to_numpy())
    return pd.DataFrame({"Hojas": trayectoria["Hojas"], "alpha": trayectoria["alpha"],
                         "Error CV": errores.mean(axis=0), "Desviación CV": errores.std(axis=0, ddof=1)})

In [ ]:
tr = arbol_entropia(X_train, y_train)
camino_poda = trayectoria_poda(tr)
camino_poda.drop(columns="podados")

#### Validación cruzada de la poda

Usamos diez particiones de entrenamiento y evaluamos las penalizaciones de la secuencia.
El eje horizontal muestra el número de hojas del árbol completo correspondiente a cada penalización.
La muestra test queda reservada para la evaluación posterior.

In [ ]:
with threadpool_limits(limits=1):
    cv_tr = validar_poda(X_train, y_train, camino_poda, cv=10)

cv_tr

In [ ]:
fig, ax = plt.subplots()
ax.plot(cv_tr["Hojas"], cv_tr["Error CV"], "o-", markersize=4)
ax.set(xlabel="Número de hojas", ylabel="Error de clasificación CV", title="Validación cruzada de la poda")
ax.grid(alpha=0.2)
plt.show()

#### Modelo podado a diez hojas

Si la secuencia no contiene exactamente diez hojas, elegimos el siguiente tamaño mayor disponible.
Representamos el árbol podado completo; el control permite inspeccionar otros tamaños de la secuencia.

In [ ]:
poda_10 = podar_a_hojas(camino_poda, numero=10)
pred_poda_10 = tr.predict(X_test, podados=poda_10)
evaluacion_poda_10 = evaluar_clasificador(y_test, pred_poda_10, clases)
evaluacion_poda_10.confusion
evaluacion_poda_10.acierto
len(tr.hojas(poda_10))
profundidad_poda_10 = max(tr.nodos[i].profundidad for i in tr.hojas(poda_10))
dibujar_arbol(tr, profundidad=profundidad_poda_10, podados=poda_10, titulo="Árbol podado a diez hojas")

In [ ]:
def explorar_poda_coste(hojas):
    podados = podar_a_hojas(camino_poda, numero=hojas)
    prediccion = tr.predict(X_test, podados=podados)
    resultado = evaluar_clasificador(y_test, prediccion, clases)
    display(pd.Series({"Hojas": len(tr.hojas(podados)), "Proporción de aciertos": resultado.acierto}))
    display(resultado.confusion)
    dibujar_arbol(tr, profundidad=3, podados=podados, titulo=f"Árbol con {hojas} hojas · primeros niveles")

control_poda_coste = widgets.interactive(explorar_poda_coste,
    hojas=widgets.Dropdown(options=sorted(camino_poda["Hojas"].unique()),
                           value=len(tr.hojas(poda_10)), description="Hojas"))
display(control_poda_coste)

## Ejercicio 7-1

Estudia cómo varía la proporción de aciertos del árbol condicional al modificar un parámetro de su
ajuste. Puedes escoger, por ejemplo, `max_depth`, `min_samples_split`, `min_samples_leaf` o `alpha`.
Conserva la partición de datos y registra los valores que pruebes y sus resultados.

In [ ]:
# 7-1. Escoger un parámetro del árbol condicional, variar sus valores y estudiar el acierto.

## 2. Random forest

Entrenamos un bosque de **50 árboles**, con remuestreo bootstrap y 9 predictores candidatos por división:
la parte entera de la raíz cuadrada de 93. Cada árbol crece sin un límite explícito de profundidad.
La predicción combina los árboles del bosque.

In [ ]:
rf_mod = RandomForestClassifier(n_estimators=50, max_features=int(np.sqrt(len(predictores))),
                                bootstrap=True, min_samples_leaf=1, random_state=SEED, n_jobs=1)
rf_mod.fit(X_train, y_train)
rf_mod

In [ ]:
pred_rf = rf_mod.predict(X_test)
evaluacion_rf = evaluar_clasificador(y_test, pred_rf, clases)
t_rf = evaluacion_rf.confusion
t_rf
evaluacion_rf.acierto

### Error fuera de bolsa: OOB

Una observación queda fuera de la muestra bootstrap de algunos árboles.
Para evaluarla por OOB usamos exclusivamente los votos de esos árboles y registramos cómo evoluciona el error
al añadir árboles. En cada punto se incluyen las filas que ya disponen de algún voto OOB.
La evaluación OOB ofrece una estimación interna; mantenemos el test para contrastar el resultado final.
[Bosques y estimación OOB](https://scikit-learn.org/1.3/modules/ensemble.html#forest).

In [ ]:
def indices_oob(arbol, n):
    muestra = np.random.RandomState(arbol.random_state).randint(0, n, n)
    return np.flatnonzero(np.bincount(muestra, minlength=n) == 0)


def diagnostico_oob(bosque, X, y):
    valores = np.asarray(X, dtype=np.float32)
    codigos = pd.Categorical(y, categories=bosque.classes_).codes
    votos = np.zeros((len(X), len(bosque.classes_)), dtype=int)
    historial = []
    for arbol in bosque.estimators_:
        fuera = indices_oob(arbol, len(X))
        prediccion = arbol.predict(valores[fuera]).astype(int)
        np.add.at(votos, (fuera, prediccion), 1)
        disponibles = votos.sum(axis=1) > 0
        ganadora = votos.argmax(axis=1)
        error = ganadora != codigos
        fila = {"OOB": error[disponibles].mean()}
        for j, clase in enumerate(bosque.classes_):
            seleccion = disponibles & (codigos == j)
            fila[clase] = error[seleccion].mean() if seleccion.any() else np.nan
        fila["Cobertura"] = disponibles.mean()
        historial.append(fila)
    disponibles = votos.sum(axis=1) > 0
    predicciones = pd.Series(bosque.classes_[votos[disponibles].argmax(axis=1)], index=X.index[disponibles])
    tabla = confusion(np.asarray(y)[disponibles], predicciones, bosque.classes_).T
    tabla = tabla.rename_axis(index="Real", columns="Predicción")
    tabla["Error de clase"] = 1-np.diag(tabla.to_numpy())/tabla.sum(axis=1)
    curva = pd.DataFrame(historial, index=np.arange(1,len(historial)+1)).rename_axis("Árboles")
    return SimpleNamespace(curva=curva, votos=pd.DataFrame(votos,index=X.index,columns=bosque.classes_),
                           confusion=tabla, predicciones=predicciones)

In [ ]:
oob_rf = diagnostico_oob(rf_mod, X_train, y_train)
oob_rf.curva.tail()
pd.Series({"Árboles": len(rf_mod.estimators_), "mtry": rf_mod.max_features,
           "Error OOB": oob_rf.curva["OOB"].iloc[-1], "Cobertura OOB": oob_rf.curva["Cobertura"].iloc[-1]})

#### Error global y por clase: test y OOB

In [ ]:
errores_test = pd.concat([pd.Series({"Global": 1-evaluacion_rf.acierto}), 1-evaluacion_rf.por_clase])
errores_oob = oob_rf.curva.loc[50, ["OOB"]+list(clases)].copy()
errores_oob.index = errores_test.index
tabla_errores_rf = pd.DataFrame({"Test": errores_test, "OOB": errores_oob})
tabla_errores_rf

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(tabla_errores_rf.index, tabla_errores_rf["Test"], "s-", label="Test")
ax.plot(tabla_errores_rf.index, tabla_errores_rf["OOB"], "s--", label="OOB")
ax.set(ylim=(0, 1), ylabel="Error de clasificación", title="Error global y por clase")
plt.setp(ax.get_xticklabels(), rotation=40, ha="right")
ax.legend()
fig.tight_layout()
plt.show()

#### Matrices de confusión

La tabla OOB tiene **clases reales en filas** y añade el error de cada clase.
Para leer ambas distribuciones de la misma forma, normalizamos las matrices con clases reales en filas.

In [ ]:
confusion_oob = oob_rf.confusion
confusion_oob
test_por_clase = t_rf.T.div(t_rf.T.sum(axis=1), axis=0)
oob_por_clase = confusion_oob.loc[:, clases].div(confusion_oob.loc[:, clases].sum(axis=1), axis=0)
test_por_clase.round(2)
oob_por_clase.round(2)

### ¿Necesitamos más árboles?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for clase in ["OOB"]+list(clases):
    ax.plot(oob_rf.curva.index, oob_rf.curva[clase], label=clase,
            linewidth=2.5 if clase == "OOB" else 1, linestyle="-" if clase == "OOB" else "--")
ax.set(xlabel="Número de árboles", ylabel="Error OOB", ylim=(0, 1), title="Evolución del error fuera de bolsa")
ax.legend(ncol=2, fontsize=8)
fig.tight_layout()
plt.show()

In [ ]:
def explorar_oob(clase, arboles):
    curva = oob_rf.curva.iloc[:arboles]
    fig, ax = plt.subplots()
    ax.plot(curva.index, curva[clase], "o-", markersize=3)
    ax.set(xlabel="Árboles", ylabel="Error OOB", ylim=(0, 1), title=f"Error OOB · {clase}")
    plt.show()
    display(curva.tail(1))

control_oob = widgets.interactive(explorar_oob,
    clase=widgets.Dropdown(options=["OOB"]+list(clases), description="Clase"),
    arboles=widgets.IntSlider(value=50, min=1, max=50, description="Árboles", continuous_update=False))
display(control_oob)

### Importancia de las variables

Para cada árbol permutamos una característica en sus observaciones OOB y medimos la pérdida de acierto,
tanto global como por clase. Promediamos las pérdidas entre árboles y las escalamos por su error estándar.
`MeanDecreaseGini` recoge la disminución total de impureza Gini, promediada entre árboles.
El cálculo conserva los resultados globales y de las nueve clases para todas las características.

In [ ]:
def importancia_oob(bosque, X, y):
    valores = np.asarray(X, dtype=np.float32)
    codigos = pd.Categorical(y, categories=bosque.classes_).codes
    n_arboles, n_variables, n_clases = len(bosque.estimators_), X.shape[1], len(bosque.classes_)
    descensos = np.zeros((n_arboles, n_variables, n_clases+1))
    gini = np.zeros((n_arboles, n_variables))
    rng = np.random.default_rng(SEED)
    for i, arbol in enumerate(bosque.estimators_):
        fuera = indices_oob(arbol, len(X))
        entrada, reales = valores[fuera].copy(), codigos[fuera]
        base = arbol.predict(entrada).astype(int) == reales
        bases = np.r_[[base[reales == c].mean() for c in range(n_clases)], base.mean()]
        for j in range(n_variables):
            original = entrada[:,j].copy()
            entrada[:,j] = rng.permutation(original)
            aciertos = arbol.predict(entrada).astype(int) == reales
            resultado = np.r_[[aciertos[reales == c].mean() for c in range(n_clases)], aciertos.mean()]
            descensos[i,j] = bases-resultado
            entrada[:,j] = original
        gini[i] = arbol.tree_.compute_feature_importances(normalize=False)*arbol.tree_.weighted_n_node_samples[0]
    medias = descensos.mean(axis=0)
    error_estandar = descensos.std(axis=0, ddof=0)/np.sqrt(n_arboles)
    escaladas = medias/np.where(error_estandar > 1e-15, error_estandar, 1)
    nombres = list(bosque.classes_)+["MeanDecreaseAccuracy"]
    tabla = pd.DataFrame(escaladas, index=X.columns, columns=nombres)
    tabla["MeanDecreaseGini"] = gini.mean(axis=0)
    return SimpleNamespace(tabla=tabla, descensos=pd.DataFrame(medias,index=X.columns,columns=nombres),
                           error_estandar=pd.DataFrame(error_estandar,index=X.columns,columns=nombres))

In [ ]:
v_imp = importancia_oob(rf_mod, X_train, y_train)
v_imp0 = v_imp.tabla
v_imp0

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 8), constrained_layout=True)
for ax, medida in zip(ejes, ["MeanDecreaseAccuracy", "MeanDecreaseGini"]):
    seleccion = v_imp0[medida].nlargest(30).sort_values()
    ax.barh(seleccion.index, seleccion)
    ax.set(title=medida, xlabel="Importancia")
plt.show()

#### Importancia global

In [ ]:
orden_global = v_imp0.sort_values("MeanDecreaseAccuracy", ascending=False)
orden_global[["MeanDecreaseAccuracy", "MeanDecreaseGini"]]

#### Importancia para Class_1

In [ ]:
orden_clase1 = v_imp0.sort_values("Class_1", ascending=False)
orden_clase1[["Class_1"]]

In [ ]:
def explorar_importancia(medida, cantidad):
    seleccion = v_imp0[medida].nlargest(cantidad)
    display(seleccion.to_frame())
    fig, ax = plt.subplots(figsize=(8, max(4, cantidad*0.22)))
    ax.barh(seleccion.index[::-1], seleccion.iloc[::-1])
    ax.set(xlabel="Importancia", title=medida)
    fig.tight_layout()
    plt.show()

control_importancia = widgets.interactive(explorar_importancia,
    medida=widgets.Dropdown(options=["MeanDecreaseAccuracy", "MeanDecreaseGini"]+list(clases), description="Medida"),
    cantidad=widgets.IntSlider(value=20, min=5, max=len(predictores), step=1,
                               description="Variables", continuous_update=False))
display(control_importancia)

### Ajustar mtry

`mtry` es el número de predictores candidatos en cada división.
Partimos de 9 y exploramos valores menores y mayores multiplicando o dividiendo por dos;
cada dirección se detiene cuando la mejora relativa del error OOB baja del 5 %.
Cada candidato utiliza 50 árboles.

Este barrido utiliza el conjunto completo, como parte del recorrido exploratorio de la sesión.
Para validar una elección definitiva, realiza también este ajuste exclusivamente con entrenamiento.

In [ ]:
def tunear_mtry(X, y, n_arboles=50, mejora_minima=0.05):
    inicio = int(np.sqrt(X.shape[1]))
    resultados = {}
    def evaluar(mtry):
        modelo = RandomForestClassifier(n_estimators=n_arboles, max_features=mtry,
                                        bootstrap=True, random_state=SEED, n_jobs=1).fit(X,y)
        diagnostico = diagnostico_oob(modelo,X,y)
        error = float(diagnostico.curva["OOB"].iloc[-1])
        resultados[mtry] = error
        return error
    error_referencia = evaluar(inicio)
    for direccion in [-1,1]:
        actual = inicio
        while error_referencia > 0:
            nuevo = max(1,int(np.ceil(actual/2))) if direccion < 0 else min(X.shape[1],actual*2)
            if nuevo == actual:
                break
            error = evaluar(nuevo)
            mejora = 1-error/error_referencia
            if mejora > mejora_minima:
                error_referencia = error
            actual = nuevo
            if mejora < mejora_minima:
                break
    return pd.Series(resultados,name="Error OOB").sort_index().rename_axis("mtry").to_frame()

In [ ]:
mtry_par = tunear_mtry(d[predictores], d["target"], n_arboles=50)
mtry_par
fig, ax = plt.subplots()
ax.plot(mtry_par.index, mtry_par["Error OOB"], "o-")
ax.set(xscale="log", xticks=mtry_par.index, xticklabels=mtry_par.index,
       xlabel="mtry", ylabel="Error OOB", title="Búsqueda de mtry")
plt.show()

#### Ajuste con mtry = 18

In [ ]:
rf_mod1 = RandomForestClassifier(n_estimators=50, max_features=18, bootstrap=True,
                                 min_samples_leaf=1, random_state=SEED, n_jobs=1).fit(X_train, y_train)
pred_rf1 = rf_mod1.predict(X_test)
evaluacion_rf1 = evaluar_clasificador(y_test, pred_rf1, clases)
evaluacion_rf1.confusion
evaluacion_rf1.acierto

#### Inspeccionar la predicción de un producto

In [ ]:
def explorar_producto_rf(ajuste, producto):
    modelo = rf_mod if ajuste == "mtry = 9" else rf_mod1
    entrada = X_test.iloc[[producto]]
    probabilidades = pd.Series(modelo.predict_proba(entrada)[0], index=modelo.classes_, name="Probabilidad")
    display(pd.Series({"Índice": entrada.index[0], "Clase real": y_test.iloc[producto],
                       "Predicción": modelo.predict(entrada)[0]}))
    display(probabilidades.to_frame())
    barras_clases(probabilidades, "Probabilidad estimada por el bosque", ylabel="Probabilidad")

control_producto_rf = widgets.interactive(explorar_producto_rf,
    ajuste=widgets.Dropdown(options=["mtry = 9", "mtry = 18"], description="Bosque"),
    producto=widgets.IntSlider(value=0, min=0, max=len(X_test)-1, description="Producto", continuous_update=False))
display(control_producto_rf)

## Ejercicio 7-2

Ajusta un random forest con **500 árboles** usando todos los datos.
¿Cuál es su error de predicción estimado mediante OOB? ¿Se necesitan más árboles?
Examina también la evolución de los errores por clase.

In [ ]:
# 7-2. Ajustar el bosque de 500 árboles y estudiar su error OOB y su evolución.

## 3. Máquinas de vectores soporte: SVM

### Submuestra y respuesta binaria

Seleccionamos 4 000 productos sin reemplazo y creamos una respuesta booleana:
**True** para `Class_5` y **False** para el resto. Cada fila pasa a entrenamiento con probabilidad 0,5.
Usamos nombres propios de este bloque para mantener activos los controles anteriores.

In [ ]:
rng_svm = np.random.default_rng(SEED)
seleccion_svm = rng_svm.choice(len(d), size=4000, replace=False)
d1 = d.iloc[seleccion_svm].copy()
d1["target2"] = d1["target"].eq("Class_5")
d1 = d1.drop(columns="target")
train_sel_svm = rng_svm.choice([False, True], size=len(d1), p=[0.5, 0.5])
train_svm = d1.loc[train_sel_svm].copy()
test_svm = d1.loc[~train_sel_svm].copy()
X_train_svm, y_train_svm = train_svm[predictores], train_svm["target2"]
X_test_svm, y_test_svm = test_svm[predictores], test_svm["target2"]
pd.Series({"Entrenamiento": len(train_svm), "Test": len(test_svm)})
pd.DataFrame({"Entrenamiento": y_train_svm.value_counts(), "Test": y_test_svm.value_counts()}).sort_index()

### Ajustar un modelo radial con C = 1

Estandarizamos las variables dentro de un `Pipeline`, usando los parámetros de entrenamiento.
`C` controla la penalización de los errores del margen. Fijamos `gamma = 1 / 93`, grado 3 y término independiente 0.
El escalado del pipeline se vuelve a ajustar dentro de cada partición de validación cruzada.
[Interfaz de SVC](https://scikit-learn.org/1.3/modules/generated/sklearn.svm.SVC.html).

In [ ]:
def construir_svm(kernel="rbf", C=1):
    return Pipeline([("escala", StandardScaler()),
                     ("svm", SVC(C=C, kernel=kernel, gamma=1/len(predictores), degree=3,
                                 coef0=0, tol=0.001, cache_size=256))])


def resumen_busqueda(busqueda):
    tabla = pd.DataFrame(busqueda.cv_results_)
    resultado = pd.DataFrame({"Parámetros": tabla["params"],
                             "Error CV": 1-tabla["mean_test_score"],
                             "Desviación CV": tabla["std_test_score"]*np.sqrt(busqueda.n_splits_/(busqueda.n_splits_-1)),
                             "Orden": tabla["rank_test_score"]})
    return resultado.sort_values("Orden",kind="stable")


def resumen_svm(modelo):
    ajuste = modelo.named_steps["svm"]
    return pd.Series({"Kernel": ajuste.kernel, "C": ajuste.C, "gamma": ajuste.gamma,
                      "Grado": ajuste.degree, "Vectores soporte": len(ajuste.support_),
                      "Soporte clase False": ajuste.n_support_[0], "Soporte clase True": ajuste.n_support_[1]})

In [ ]:
mod_svm = construir_svm(kernel="rbf", C=1).fit(X_train_svm, y_train_svm)
resumen_svm(mod_svm)

### Capacidad predictiva

In [ ]:
pr_svm = mod_svm.predict(X_test_svm)
evaluacion_svm = evaluar_clasificador(y_test_svm, pr_svm, [False, True])
evaluacion_svm.acierto
evaluacion_svm.confusion

### Ajustar C con un kernel lineal

Probamos los siete costes del guion mediante **validación cruzada de diez particiones** sobre entrenamiento.
Se conserva el orden propuesto de candidatos; la selección maximiza el acierto medio.
Después se reajusta el mejor candidato con todo el entrenamiento.

In [ ]:
costes = [0.01, 0.2, 0.1, 1, 5, 10, 100]
cv_svm = KFold(n_splits=10, shuffle=True, random_state=SEED)
mod_tune = GridSearchCV(construir_svm("linear"), {"svm__C": costes},
                        cv=cv_svm, scoring="accuracy", refit=True, n_jobs=1, error_score="raise")
with threadpool_limits(limits=1):
    mod_tune.fit(X_train_svm, y_train_svm)

resumen_busqueda(mod_tune)
mod_tune.best_params_

In [ ]:
mod_best = mod_tune.best_estimator_
resumen_svm(mod_best)

#### Evaluar el modelo lineal elegido

In [ ]:
pr_lineal = mod_best.predict(X_test_svm)
evaluacion_lineal = evaluar_clasificador(y_test_svm, pr_lineal, [False, True])
evaluacion_lineal.confusion
evaluacion_lineal.acierto

### Kernel polinómico

In [ ]:
mod_tune1 = GridSearchCV(construir_svm("poly"), {"svm__C": costes},
                         cv=cv_svm, scoring="accuracy", refit=True, n_jobs=1, error_score="raise")
with threadpool_limits(limits=1):
    mod_tune1.fit(X_train_svm, y_train_svm)

resumen_busqueda(mod_tune1)
mod_best1 = mod_tune1.best_estimator_
resumen_svm(mod_best1)

In [ ]:
pr_polinomico = mod_best1.predict(X_test_svm)
evaluacion_polinomica = evaluar_clasificador(y_test_svm, pr_polinomico, [False, True])
evaluacion_polinomica.confusion
evaluacion_polinomica.acierto

### Kernel radial

In [ ]:
mod_tune2 = GridSearchCV(construir_svm("rbf"), {"svm__C": costes},
                         cv=cv_svm, scoring="accuracy", refit=True, n_jobs=1, error_score="raise")
with threadpool_limits(limits=1):
    mod_tune2.fit(X_train_svm, y_train_svm)

resumen_busqueda(mod_tune2)
mod_best2 = mod_tune2.best_estimator_
resumen_svm(mod_best2)

In [ ]:
pr_radial = mod_best2.predict(X_test_svm)
evaluacion_radial = evaluar_clasificador(y_test_svm, pr_radial, [False, True])
evaluacion_radial.confusion
evaluacion_radial.acierto

### ¿Qué kernel escoger con C = 1?

Probamos lineal, polinómico, radial y sigmoide, conservando el coste en 1 y las mismas diez particiones.

In [ ]:
mod_tune3 = GridSearchCV(construir_svm(C=1), {"svm__kernel": ["linear", "poly", "rbf", "sigmoid"]},
                         cv=cv_svm, scoring="accuracy", refit=True, n_jobs=1, error_score="raise")
with threadpool_limits(limits=1):
    mod_tune3.fit(X_train_svm, y_train_svm)

resumen_busqueda(mod_tune3)
mod_tune3.best_params_
mod_best3 = mod_tune3.best_estimator_
resumen_svm(mod_best3)

In [ ]:
pr_kernel = mod_best3.predict(X_test_svm)
evaluacion_kernel = evaluar_clasificador(y_test_svm, pr_kernel, [False, True])
evaluacion_kernel.confusion
evaluacion_kernel.acierto

#### Explorar la validación cruzada

Las barras muestran el error medio y su desviación entre particiones, sin utilizar las etiquetas de test.

In [ ]:
busquedas_svm = {"Lineal: C": mod_tune, "Polinómico: C": mod_tune1,
                 "Radial: C": mod_tune2, "Kernel: C = 1": mod_tune3}

def explorar_cv_svm(busqueda):
    ajuste = busquedas_svm[busqueda]
    tabla = resumen_busqueda(ajuste)
    display(tabla, ajuste.best_params_)
    parametro = "svm__kernel" if ajuste is mod_tune3 else "svm__C"
    etiquetas = [str(p[parametro]) for p in ajuste.cv_results_["params"]]
    error = 1-ajuste.cv_results_["mean_test_score"]
    desviacion = ajuste.cv_results_["std_test_score"]*np.sqrt(10/9)
    fig, ax = plt.subplots()
    ax.bar(etiquetas, error, yerr=desviacion, capsize=3)
    ax.set(xlabel="Kernel" if parametro.endswith("kernel") else "C",
           ylabel="Error CV", title=busqueda)
    plt.show()

control_cv_svm = widgets.interactive(explorar_cv_svm,
    busqueda=widgets.Dropdown(options=list(busquedas_svm), description="Búsqueda"))
display(control_cv_svm)

#### Explorar la función de decisión

Elegimos dos características para representar un **corte del espacio de predictores**, fijando las demás
en sus medianas de entrenamiento. El color de fondo representa el signo de la función de decisión;
los puntos sitúan una muestra reproducible de productos de test.
La clasificación y la matriz de confusión utilizan los 93 predictores reales de cada producto.

In [ ]:
modelos_svm = {"Radial inicial": mod_svm, "Lineal elegido": mod_best, "Polinómico elegido": mod_best1,
               "Radial elegido": mod_best2, "Kernel elegido con C=1": mod_best3}

def explorar_decision_svm(modelo, eje_x, eje_y):
    if eje_x == eje_y:
        display("Selecciona dos características distintas.")
        return
    ajuste = modelos_svm[modelo]
    prediccion = ajuste.predict(X_test_svm)
    resultado = evaluar_clasificador(y_test_svm, prediccion, [False, True])
    display(resumen_svm(ajuste), resultado.confusion, pd.Series({"Proporción de aciertos": resultado.acierto}))
    limites = []
    for variable in [eje_x, eje_y]:
        a, b = X_train_svm[variable].quantile([0.01, 0.99])
        if a == b:
            a, b = X_train_svm[variable].min(), X_train_svm[variable].max()
        if a == b:
            a, b = a-1, b+1
        margen = (b-a)*0.05
        limites.append((a-margen, b+margen))
    xx, yy = np.meshgrid(np.linspace(*limites[0], 70), np.linspace(*limites[1], 70))
    rejilla = pd.DataFrame(np.tile(X_train_svm.median().to_numpy(), (xx.size, 1)), columns=predictores)
    rejilla[eje_x], rejilla[eje_y] = xx.ravel(), yy.ravel()
    superficie = ajuste.decision_function(rejilla).reshape(xx.shape)
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.contourf(xx, yy, (superficie > 0).astype(int), levels=[-0.5, 0.5, 1.5],
                colors=["#d7e9fb", "#fbdcca"], alpha=0.8)
    if superficie.min() < 0 < superficie.max():
        ax.contour(xx, yy, superficie, levels=[0], colors="black", linewidths=1)
    muestra = np.random.default_rng(SEED).choice(len(X_test_svm), size=min(600,len(X_test_svm)), replace=False)
    for clase, color in [(False,"steelblue"), (True,"darkorange")]:
        seleccion = muestra[y_test_svm.iloc[muestra].to_numpy() == clase]
        ax.scatter(X_test_svm.iloc[seleccion][eje_x], X_test_svm.iloc[seleccion][eje_y],
                   s=16, alpha=.5, c=color, label=f"Clase real: {clase}")
    ax.set(xlim=limites[0], ylim=limites[1], xlabel=eje_x, ylabel=eje_y,
           title=f"{modelo} · corte con las otras variables en su mediana")
    ax.legend()
    fig.tight_layout()
    plt.show()

control_decision_svm = widgets.interactive(explorar_decision_svm,
    modelo=widgets.Dropdown(options=list(modelos_svm), value="Lineal elegido", description="Modelo"),
    eje_x=widgets.Dropdown(options=predictores, value="feat_34", description="Eje X"),
    eje_y=widgets.Dropdown(options=predictores, value="feat_83", description="Eje Y"))
display(control_decision_svm)

## Ejercicio 7-3

Mejora la capacidad predictiva de una SVM para clasificar **todas las clases** del producto.
Utiliza una búsqueda de hiperparámetros con `GridSearchCV` y prueba con una submuestra ligeramente mayor.
Reserva test para evaluar la configuración final y ajusta el escalado dentro de la validación cruzada.

In [ ]:
# 7-3. Ampliar la submuestra, recuperar las nueve clases y ajustar los hiperparámetros de la SVM.